## Student ID: 2500575

In [13]:
# INSTALL LIBRARIES
# ==========================================================

!pip install -q transformers sentencepiece accelerate

In [22]:
# IMPORTS
# ==========================================================

import os
import pandas as pd
import numpy as np
import joblib
import torch
import warnings

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score

from transformers import pipeline

warnings.filterwarnings("ignore")


# ==========================================================
# STUDENT ID
# ==========================================================

student_id = 2500575


# ==========================================================
# SET SEED
# ==========================================================

np.random.seed(student_id)


# ==========================================================
# MOUNT GOOGLE DRIVE
# ==========================================================

from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)


# ==========================================================
# GOOGLE DRIVE PATH
# ==========================================================

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'CE807-26-SP/Assignment/'

GOOGLE_DRIVE_PATH = os.path.join(
    'gdrive',
    'MyDrive',
    GOOGLE_DRIVE_PATH_AFTER_MYDRIVE
)

print('List files:', os.listdir(GOOGLE_DRIVE_PATH))


# ==========================================================
# DATA PATH
# ==========================================================

DATA_PATH = os.path.join(
    GOOGLE_DRIVE_PATH,
    'data',
    '75'
)

train_file = os.path.join(
    DATA_PATH,
    'train.csv'
)

val_file = os.path.join(
    DATA_PATH,
    'valid.csv'
)

test_file = os.path.join(
    DATA_PATH,
    'test.csv'
)

print("Train file:", train_file)
print("Validation file:", val_file)
print("Test file:", test_file)


# ==========================================================
# READ DATA
# ==========================================================

train_df = pd.read_csv(train_file)
train_df.head()

Mounted at /content/gdrive
List files: ['model', 'data']
Train file: gdrive/MyDrive/CE807-26-SP/Assignment/data/75/train.csv
Validation file: gdrive/MyDrive/CE807-26-SP/Assignment/data/75/valid.csv
Test file: gdrive/MyDrive/CE807-26-SP/Assignment/data/75/test.csv


,sentiment,text,data_id
0,positive,"Works in my 30"" BlueStar range. My model # is...",75
1,positive,Perfect for the dryer i have.,75
2,positive,"Works great, easy to install, saves tons of space",75
3,positive,When our washer didn’t drain it was YouTube to...,75
4,positive,"Very good product, worked as advertised",75


In [14]:
val_df = pd.read_csv(val_file)
val_df.head()

,sentiment,text,data_id
0,negative,I tired of things falling between counter & re...,75
1,positive,This is an exact replacement of the part that ...,75
2,positive,Perfect!!,75
3,negative,Broke editing instillation,75
4,positive,This was for a Kitchenaid dishwasher. It appe...,75


In [15]:
test_df = pd.read_csv(test_file)


In [16]:
# ==========================================================
# READ DATA FUNCTION
# ==========================================================

def read_data(file_name):

    df = pd.read_csv(file_name)

    print(
        file_name,
        'has',
        len(df),
        'data points'
    )

    return df


test_df = read_data(test_file)

gdrive/MyDrive/CE807-26-SP/Assignment/data/75/test.csv has 1386 data points


In [17]:
# ==========================================================
# MODEL PATH
# ==========================================================

MODEL_PATH = os.path.join(
    GOOGLE_DRIVE_PATH,
    'model',
    str(student_id)
)

MODEL_Dis_DIRECTORY = os.path.join(
    MODEL_PATH,
    'model_dis'
)

MODEL_unsup_DIRECTORY = os.path.join(
    MODEL_PATH,
    'model_unsup'
)

print(
    'MLP Model directory:',
    MODEL_Dis_DIRECTORY
)

print(
    'LLM directory:',
    MODEL_unsup_DIRECTORY
)

os.makedirs(
    MODEL_Dis_DIRECTORY,
    exist_ok=True
)

os.makedirs(
    MODEL_unsup_DIRECTORY,
    exist_ok=True
)

MLP Model directory: gdrive/MyDrive/CE807-26-SP/Assignment/model/2500575/model_dis
LLM directory: gdrive/MyDrive/CE807-26-SP/Assignment/model/2500575/model_unsup


In [18]:
# ==========================================================
# COLUMN NAMES
# ==========================================================

TEXT_COL = 'text'
LABEL_COL = 'sentiment'


# ==========================================================
# TRAIN FUNCTION
# MODEL 1: MLP
# ==========================================================

def train_Gen(
    train_file,
    val_file,
    model_dir
):

    # Read files
    train_df = pd.read_csv(train_file)
    val_df = pd.read_csv(val_file)

    # Handle missing text
    train_df[TEXT_COL] = (
        train_df[TEXT_COL]
        .fillna("")
        .astype(str)
    )

    val_df[TEXT_COL] = (
        val_df[TEXT_COL]
        .fillna("")
        .astype(str)
    )

    # Convert labels
    train_df[LABEL_COL] = (
        train_df[LABEL_COL]
        .map({
            'positive': 1,
            'negative': 0
        })
    )

    val_df[LABEL_COL] = (
        val_df[LABEL_COL]
        .map({
            'positive': 1,
            'negative': 0
        })
    )

    print(
        "Training size:",
        len(train_df)
    )

    # ======================================================
    # MLP + TF-IDF
    # ======================================================

    model = Pipeline([

        (
            'tfidf',
            TfidfVectorizer(
                max_features=6000,
                ngram_range=(1, 2),
                stop_words='english'
            )
        ),

        (
            'clf',
            MLPClassifier(
                hidden_layer_sizes=(100,),
                max_iter=100,
                random_state=student_id,
                early_stopping=True
            )
        )

    ])

    print(
        "Training MLP..."
    )

    model.fit(
        train_df[TEXT_COL],
        train_df[LABEL_COL]
    )

    # ======================================================
    # VALIDATION
    # ======================================================

    val_pred = model.predict(
        val_df[TEXT_COL]
    )

    score = f1_score(
        val_df[LABEL_COL],
        val_pred,
        average='macro'
    )

    print(
        "MLP Validation F1 Score:",
        score
    )

    # ======================================================
    # SAVE MODEL
    # ======================================================

    joblib.dump(
        model,
        os.path.join(
            model_dir,
            'mlp_model.pkl'
        )
    )

    print(
        "MLP model saved"
    )

In [19]:
# ==========================================================
# TEST FUNCTION
# MODEL 1: MLP
# MODEL 2: OPEN-SOURCE LLM + 5 PROMPTS
# ==========================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


def test_Gen(
    test_file,
    model_dir
):

    # ======================================================
    # READ TEST DATA
    # ======================================================

    test_df = pd.read_csv(test_file)

    test_df[TEXT_COL] = (
        test_df[TEXT_COL]
        .fillna("")
        .astype(str)
    )

    print(
        "Testing size:",
        len(test_df)
    )


    # ======================================================
    # LOAD MLP MODEL
    # ======================================================

    model = joblib.load(
        os.path.join(
            model_dir,
            'mlp_model.pkl'
        )
    )

    print(
        "MLP model loaded successfully"
    )


    # ======================================================
    # MLP PREDICTION
    # ======================================================

    test_df[
        'out_label_MLP'
    ] = model.predict(
        test_df[TEXT_COL]
    )

    print(
        "MLP predictions completed"
    )


    # ======================================================
    # LOAD OPEN-SOURCE LLM
    # ======================================================

    print(
        "\nLoading LLM..."
    )

    MODEL_NAME = "google/flan-t5-small"


    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )


    # Load model
    llm = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME
    )


    # GPU if available
    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    llm = llm.to(device)

    llm.eval()


    print(
        "LLM loaded successfully"
    )

    print(
        "Device:",
        device
    )


    # ======================================================
    # SAVE LLM INFORMATION
    # ======================================================

    os.makedirs(
        MODEL_unsup_DIRECTORY,
        exist_ok=True
    )


    with open(
        os.path.join(
            MODEL_unsup_DIRECTORY,
            "llm_info.txt"
        ),
        "w"
    ) as f:

        f.write(
            "Model: google/flan-t5-small\n"
        )

        f.write(
            "Task: Zero-shot sentiment classification\n"
        )

        f.write(
            "Number of prompts: 5\n"
        )


    # ======================================================
    # FIVE PROMPTS
    # ======================================================

    PROMPTS = {

        1:
        "Classify this text as positive or negative: {}",

        2:
        "Is the sentiment of this text positive or negative? {}",

        3:
        "Analyze the sentiment. Answer only positive or negative: {}",

        4:
        "Sentiment classification task. Choose positive or negative: {}",

        5:
        "Determine whether the following text expresses positive or negative sentiment: {}"

    }


    # ======================================================
    # RUN ALL 5 PROMPTS
    # ======================================================

    for i in range(1, 6):

        print(
            f"\nRunning Prompt {i}..."
        )


        preds = []


        # ==================================================
        # PROCESS EACH TEST TEXT
        # ==================================================

        for number, text in enumerate(
            test_df[TEXT_COL]
        ):


            # Limit very long text
            text = text[:500]


            # Create prompt
            prompt_text = (
                PROMPTS[i]
                .format(text)
            )


            # ==============================================
            # TOKENIZE PROMPT
            # ==============================================

            inputs = tokenizer(
                prompt_text,
                return_tensors="pt",
                truncation=True,
                max_length=512
            )


            # Move input to GPU/CPU
            inputs = {

                key: value.to(device)

                for key, value
                in inputs.items()

            }


            # ==============================================
            # GENERATE LLM ANSWER
            # ==============================================

            with torch.no_grad():

                outputs = llm.generate(
                    **inputs,
                    max_new_tokens=5,
                    do_sample=False
                )


            # ==============================================
            # DECODE ANSWER
            # ==============================================

            answer = tokenizer.decode(
                outputs[0],
                skip_special_tokens=True
            )


            answer = (
                answer
                .lower()
                .strip()
            )


            # ==============================================
            # CONVERT ANSWER TO LABEL
            #
            # Positive = 1
            # Negative = 0
            # ==============================================

            if "positive" in answer:

                preds.append(1)

            elif "negative" in answer:

                preds.append(0)

            else:

                # Fallback if model gives
                # unexpected answer
                preds.append(0)


            # Show progress every 50 rows
            if (
                (number + 1) % 50 == 0
                or
                (number + 1) == len(test_df)
            ):

                print(
                    f"Processed "
                    f"{number + 1}/"
                    f"{len(test_df)}"
                )


        # ==================================================
        # SAVE PROMPT PREDICTIONS
        # ==================================================

        test_df[
            f'out_label_Prompt_{i}'
        ] = preds


        print(
            f"Prompt {i} completed"
        )


    # ======================================================
    # CHECK REQUIRED OUTPUT COLUMNS
    # ======================================================

    required_columns = [

        'out_label_MLP',

        'out_label_Prompt_1',

        'out_label_Prompt_2',

        'out_label_Prompt_3',

        'out_label_Prompt_4',

        'out_label_Prompt_5'

    ]


    print(
        "\nChecking required columns..."
    )


    for column in required_columns:

        if column in test_df.columns:

            print(
                column,
                "✓"
            )

        else:

            print(
                column,
                "MISSING"
            )


    # ======================================================
    # SAVE FINAL TEST.CSV
    # ======================================================

    test_df.to_csv(
        test_file,
        index=False
    )


    print(
        "\n========================================"
    )

    print(
        "test.csv saved successfully"
    )

    print(
        "========================================"
    )


    print(
        "\nSaved at:"
    )

    print(
        test_file
    )


    print(
        "\nFinal columns:"
    )

    print(
        test_df.columns.tolist()
    )


    # ======================================================
    # DISPLAY FIRST FIVE RESULTS
    # ======================================================

    display(
        test_df.head()
    )


    return test_df

In [20]:
# ==========================================================
# LLM VALIDATION
# F1 SCORE FOR ALL 5 PROMPTS
# ==========================================================

from sklearn.metrics import f1_score


def validate_LLM(val_file):

    # ======================================================
    # READ VALIDATION DATA
    # ======================================================

    val_df = pd.read_csv(val_file)

    val_df[TEXT_COL] = (
        val_df[TEXT_COL]
        .fillna("")
        .astype(str)
    )


    # Convert labels to 0 and 1
    val_df[LABEL_COL] = (
        val_df[LABEL_COL]
        .map({
            'positive': 1,
            'negative': 0
        })
    )

    print(
        "Validation size:",
        len(val_df)
    )


    # ======================================================
    # LOAD LLM
    # ======================================================

    print("\nLoading LLM...")

    MODEL_NAME = "google/flan-t5-small"

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    llm = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    llm = llm.to(device)

    llm.eval()

    print("LLM loaded successfully")


    # ======================================================
    # FIVE PROMPTS
    # ======================================================

    PROMPTS = {

        1:
        "Classify this text as positive or negative: {}",

        2:
        "Is the sentiment of this text positive or negative? {}",

        3:
        "Analyze the sentiment. Answer only positive or negative: {}",

        4:
        "Sentiment classification task. Choose positive or negative: {}",

        5:
        "Determine whether the following text expresses positive or negative sentiment: {}"

    }


    # ======================================================
    # TEST EACH PROMPT
    # ======================================================

    f1_results = []


    for i in range(1, 6):

        print(
            f"\nRunning Prompt {i}..."
        )

        preds = []


        for number, text in enumerate(
            val_df[TEXT_COL]
        ):

            # Limit long text
            text = text[:500]


            # Create prompt
            prompt_text = (
                PROMPTS[i]
                .format(text)
            )


            # Tokenize
            inputs = tokenizer(
                prompt_text,
                return_tensors="pt",
                truncation=True,
                max_length=512
            )


            inputs = {
                key: value.to(device)
                for key, value
                in inputs.items()
            }


            # Generate prediction
            with torch.no_grad():

                outputs = llm.generate(
                    **inputs,
                    max_new_tokens=5,
                    do_sample=False
                )


            # Decode prediction
            answer = tokenizer.decode(
                outputs[0],
                skip_special_tokens=True
            )

            answer = (
                answer
                .lower()
                .strip()
            )


            # Convert to numerical label
            if "positive" in answer:

                preds.append(1)

            elif "negative" in answer:

                preds.append(0)

            else:

                preds.append(0)


            # Progress
            if (
                (number + 1) % 50 == 0
                or
                (number + 1) == len(val_df)
            ):

                print(
                    f"Processed "
                    f"{number + 1}/"
                    f"{len(val_df)}"
                )


        # ==================================================
        # CALCULATE F1
        # ==================================================

        score = f1_score(
            val_df[LABEL_COL],
            preds,
            average='macro'
        )


        print(
            f"Prompt {i} F1 Score: "
            f"{score:.4f}"
        )


        f1_results.append({
            'Prompt': f'Prompt {i}',
            'F1 Score': score
        })


    # ======================================================
    # RESULTS
    # ======================================================

    results_df = pd.DataFrame(
        f1_results
    )


    print(
        "\n======================================"
    )

    print(
        "LLM PROMPT F1 RESULTS"
    )

    print(
        "======================================"
    )


    print(
        results_df.to_string(index=False)
    )


    # ======================================================
    # BEST PROMPT
    # ======================================================

    best_row = results_df.loc[
        results_df['F1 Score'].idxmax()
    ]


    print(
        "\nBest Prompt:",
        best_row['Prompt']
    )

    print(
        "Best F1 Score:",
        round(
            best_row['F1 Score'],
            4
        )
    )


    return results_df

In [21]:
# ==========================================================
# RUN PIPELINE
# ==========================================================

# Train MLP
train_Gen(
    train_file,
    val_file,
    MODEL_Dis_DIRECTORY
)


# Generate test predictions
test_df = test_Gen(
    test_file,
    MODEL_Dis_DIRECTORY
)


# Calculate LLM F1 scores on validation data
llm_results = validate_LLM(
    val_file
)

Training size: 4914
Training MLP...
MLP Validation F1 Score: 0.766027843813258
MLP model saved
Testing size: 1386
MLP model loaded successfully
MLP predictions completed

Loading LLM...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

,text,data_id,out_label_model_1,out_label_model_2,out_label_MLP,out_label_Prompt_1,out_label_Prompt_2,out_label_Prompt_3,out_label_Prompt_4,out_label_Prompt_5
0,Pretty Impressed I could fix my Maytag washer ...,75,NaN,NaN,1,0,0,0,0,0
1,Great product for under the sink,75,NaN,NaN,1,1,1,1,1,1
2,Replace your refrigerator filter every 6 month...,75,NaN,NaN,1,1,1,1,1,1
3,My husband YouTube’d how to fix my drier when ...,75,NaN,NaN,1,1,1,1,1,1
4,Part fit perfectly. You might have to fiddle w...,75,NaN,NaN,1,1,1,1,1,1


Validation size: 700

Loading LLM...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded successfully

Running Prompt 1...
Processed 50/700
Processed 100/700
Processed 150/700
Processed 200/700
Processed 250/700
Processed 300/700
Processed 350/700
Processed 400/700
Processed 450/700
Processed 500/700
Processed 550/700
Processed 600/700
Processed 650/700
Processed 700/700
Prompt 1 F1 Score: 0.8451

Running Prompt 2...
Processed 50/700
Processed 100/700
Processed 150/700
Processed 200/700
Processed 250/700
Processed 300/700
Processed 350/700
Processed 400/700
Processed 450/700
Processed 500/700
Processed 550/700
Processed 600/700
Processed 650/700
Processed 700/700
Prompt 2 F1 Score: 0.8484

Running Prompt 3...
Processed 50/700
Processed 100/700
Processed 150/700
Processed 200/700
Processed 250/700
Processed 300/700
Processed 350/700
Processed 400/700
Processed 450/700
Processed 500/700
Processed 550/700
Processed 600/700
Processed 650/700
Processed 700/700
Prompt 3 F1 Score: 0.8277

Running Prompt 4...
Processed 50/700
Processed 100/700
Processed 150/700
Processe